# 06. Batch, Torch, 그리고 다음 병목

지금까지는 하나의 operating point를 빠르게 푸는 관점이었다. 실제 응용에서는 같은 topology에서 load/generation scenario만 바뀌는 문제가 많다.

- contingency analysis
- time-series simulation
- Monte Carlo sampling
- optimization 또는 learning loop 안의 differentiable power flow

이때 중요한 것은 단일 solve latency뿐 아니라 batch throughput과 gradient interface다.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'cuPF').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from python.tutorial import tutorial_utils as tu

plt.rcParams['figure.figsize'] = (8, 4.8)
plt.rcParams['axes.grid'] = False
pd.set_option('display.max_colwidth', 120)

RUN_EXAMPLES = True


`solve_batch`는 같은 `Ybus`/PV/PQ 구조에서 여러 `Sbus`, `V0`를 batch-major 배열로 넘긴다. 현재 CPU path는 batch size 1 중심이고, batch > 1은 CUDA FP32/Mixed path에서 의미가 있다. 따라서 CUDA가 있으면 GPU build를 먼저 잡고, 없으면 API 모양만 확인한다.


In [2]:
try:
    import torch
    has_cuda = torch.cuda.is_available()
except Exception:
    has_cuda = False

case = tu.load_case('case9')
cupf = None
kind = 'gpu' if has_cuda else 'cpu'
if has_cuda:
    build = tu.build_eval('gpu', jobs=2, timeout=3600)
    print(tu.command_summary(build, tail_lines=14))
    cupf, msg = tu.import_cupf_from_build('gpu')
else:
    build = tu.build_eval('cpu', jobs=2, timeout=2400)
    print(tu.command_summary(build, tail_lines=14))
    cupf, msg = tu.import_cupf_from_build('cpu')
print(msg)


$ bash benchmark/scripts/build_eval.bash gpu --jobs 2
[OK] elapsed=0.3s
-- Configuring done
-- Generating done
-- Build files have been written to: /workspace/gpu-powerflow-master/cuPF/build/eval-gpu
[build_eval] build _cupf + cupf_cpp_evaluate (-j 2)
Consolidate compiler generated dependencies of target cupf
[ 89%] Built target cupf
Consolidate compiler generated dependencies of target _cupf
[100%] Built target _cupf
[ 86%] Built target cupf
Consolidate compiler generated dependencies of target cupf_cpp_evaluate
[100%] Built target cupf_cpp_evaluate
[build_eval] done. artifacts under /workspace/gpu-powerflow-master/cuPF/build/eval-gpu
[build_eval]   /workspace/gpu-powerflow-master/cuPF/build/eval-gpu/tests/cupf_cpp_evaluate
[build_eval]   /workspace/gpu-powerflow-master/cuPF/build/eval-gpu/_cupf.cpython-310-x86_64-linux-gnu.so
imported from eval-gpu


In [3]:
if cupf is None:
    print('cuPF Python module is unavailable, so the batch example cannot run.')
else:
    try:
        opts = cupf.NewtonOptions()
        if kind == 'gpu':
            opts.backend = cupf.BackendKind.CUDA
            opts.compute = cupf.ComputePolicy.Mixed
            scales = np.array([1.00, 1.01, 0.99, 1.02])
        else:
            opts.backend = cupf.BackendKind.CPU
            opts.compute = cupf.ComputePolicy.FP64
            opts.cpu_linear_solver = cupf.CpuLinearSolverKind.KLU
            scales = np.array([1.00])
        solver = cupf.NewtonSolver(opts)
        solver.initialize(case.ybus.indptr, case.ybus.indices, case.ybus.data, case.ybus.shape[0], case.ybus.shape[1], case.pv, case.pq)
        sbus_batch = np.stack([case.sbus * scale for scale in scales]).astype(np.complex128)
        v0_batch = np.stack([case.v0 for _ in scales]).astype(np.complex128)
        config = cupf.NRConfig()
        config.tolerance = 1e-6 if kind == 'gpu' else 1e-8
        config.max_iter = 50
        result = solver.solve_batch(
            case.ybus.indptr, case.ybus.indices, case.ybus.data,
            case.ybus.shape[0], case.ybus.shape[1],
            sbus_batch, v0_batch, case.pv, case.pq, config,
        )
        display(pd.DataFrame({
            'scenario': np.arange(result.batch_size),
            'load_scale': scales,
            'backend': kind,
            'converged': result.converged_numpy.astype(bool),
            'iterations': result.iterations_numpy,
            'final_mismatch': result.final_mismatch_numpy,
        }))
    except Exception as exc:
        print(f'batch solve unavailable in this configuration: {type(exc).__name__}: {exc}')


,scenario,load_scale,backend,converged,iterations,final_mismatch
0,0,1.00,gpu,True,4,3.424897e-07
1,1,1.01,gpu,True,4,3.515191e-07
2,2,0.99,gpu,True,4,3.323748e-07
3,3,1.02,gpu,True,4,3.641224e-07


Torch autograd는 batch solve와 다른 질문에 답한다. load perturbation이 voltage와 downstream loss에 어떤 영향을 주는지 gradient로 얻고 싶을 때 사용한다. 내부적으로는 forward Newton solve와 adjoint solve가 연결된다.


In [4]:
try:
    import torch
    if not torch.cuda.is_available():
        print('Torch autograd example needs CUDA.')
    else:
        cupf_gpu, _ = tu.import_cupf_from_build('gpu')
        if cupf_gpu is None or getattr(cupf_gpu, 'solve', None) is None or not hasattr(cupf_gpu.NewtonSolver, 'solve_with_adjoint_cache_torch'):
            print('cuPF was built without the Torch extension methods in this environment.')
        else:
            opts = cupf_gpu.NewtonOptions()
            opts.backend = cupf_gpu.BackendKind.CUDA
            opts.compute = cupf_gpu.ComputePolicy.Mixed
            solver = cupf_gpu.NewtonSolver(opts)
            solver.initialize(case.ybus.indptr, case.ybus.indices, case.ybus.data, case.ybus.shape[0], case.ybus.shape[1], case.pv, case.pq)
            device = torch.device('cuda')
            dtype = torch.float32
            load_p = torch.zeros((2, case.ybus.shape[0]), device=device, dtype=dtype, requires_grad=True)
            load_q = torch.zeros((2, case.ybus.shape[0]), device=device, dtype=dtype, requires_grad=True)
            load_p.data[1] = 0.01
            sbus_base_re = torch.tensor(case.sbus.real, device=device, dtype=dtype)
            sbus_base_im = torch.tensor(case.sbus.imag, device=device, dtype=dtype)
            v0_va = torch.angle(torch.tensor(case.v0, device=device, dtype=torch.complex64)).to(dtype)
            v0_vm = torch.abs(torch.tensor(case.v0, device=device, dtype=torch.complex64)).to(dtype)
            config = cupf_gpu.NRConfig(); config.tolerance = 1e-6; config.max_iter = 30
            va, vm = cupf_gpu.solve(load_p, load_q, solver, sbus_base_re=sbus_base_re, sbus_base_im=sbus_base_im, v0_va=v0_va, v0_vm=v0_vm, config=config, solve_options=cupf_gpu.SolveOptions())
            loss = va[:, 1].sum() + vm[:, 1].sum()
            loss.backward()
            display(pd.DataFrame([{'va_shape': tuple(va.shape), 'vm_shape': tuple(vm.shape), 'grad_p_finite': bool(torch.isfinite(load_p.grad).all()), 'grad_q_finite': bool(torch.isfinite(load_q.grad).all())}]))
except Exception as exc:
    print(f'Torch autograd example unavailable: {type(exc).__name__}: {exc}')


cuPF was built without the Torch extension methods in this environment.


남은 연구 방향은 앞 노트북의 결과와 직접 연결된다.

| 방향 | 앞에서 본 병목과의 연결 |
|---|---|
| custom linear solver | cuDSS가 지배적인 비용이면 전력계통 Jacobian 구조를 더 이용해야 한다. |
| cuGraph / graph scheduling | Edge/Vertex Jacobian fill은 결국 graph traversal와 sparse scatter/gather 문제다. |
| multi-GPU | 단일 solve보다 많은 scenario batch 처리에서 자연스럽다. |
| tensor core / mixed precision | FP64 비용을 줄일 수 있지만 residual correction과 수렴 안정성이 같이 필요하다. |
| iterative refinement | mixed precision solve를 Newton tolerance와 연결하는 안전장치다. |

전체 흐름을 다시 쓰면 이렇다. 전력망은 `Ybus`와 `Sbus`로 바뀌고, Newton은 mismatch와 Jacobian을 반복해서 만든다. baseline은 이 구조를 신뢰할 수 있는 기준으로 제공한다. cuPF는 같은 구조에서 sparse pattern, solver backend, GPU batch라는 반복성을 더 강하게 이용하려는 시도다.
